In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv("fra.txt",sep="\t",header=None)

In [ ]:
df.head()

In [ ]:
df=df.iloc[:,:2]
df.columns=["english","french"]

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df = df.sample(5000, random_state=42)

In [ ]:
# cleaning
import re
import string


In [ ]:
def cleaning(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z?.!,¿]+", " ", text)
  return text.strip()

In [ ]:
df["english"]=df["english"].apply(cleaning)
df["french"]=df["french"].apply(cleaning)

In [ ]:
df

In [ ]:
df["french"] = df["french"].apply(lambda x: "<start> " + x + " <end>")

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
tok = Tokenizer(filters='')
tok.fit_on_texts(df["english"])

In [ ]:
fr_tok = Tokenizer(filters='')
fr_tok.fit_on_texts(df["french"])

In [ ]:
eng_seq = tok.texts_to_sequences(df["english"])
fr_seq = fr_tok.texts_to_sequences(df["french"])

In [ ]:
max_len_eng = max(len(seq) for seq in eng_seq)
max_len_fr = max(len(seq) for seq in fr_seq)

eng_seq = pad_sequences(eng_seq, maxlen=max_len_eng, padding='post')
fr_seq = pad_sequences(fr_seq, maxlen=max_len_fr, padding='post')

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(eng_seq,fr_seq,test_size=0.1)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from keras.callbacks import EarlyStopping

embedding_dim = 256
units = 512

# Encoder
encoder_inputs = Input(shape=(max_len_eng,))
enc_emb = Embedding(len(tok.word_index)+1, embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(units, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_len_fr,))
dec_emb = Embedding(len(fr_tok.word_index)+1, embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(len(fr_tok.word_index)+1, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
model.fit([X_train, y_train], y_train, epochs=50, batch_size=32, validation_data=([X_test, y_test], y_test),callbacks=[EarlyStopping(patience=3)])

In [ ]:
loss,accuracy=model.evaluate([X_test,y_test],y_test)